###
    We will try to check and visualize a hypothetical correlation between the differential of the 2 Year US and German Bond yield and the EUR/USD.
    The differential between the both 2 years Bond Yields is called also – Yield spread.
    We will work on the period from 24/03/2016 till 24/03/2026.
###

In [195]:
import numpy as np
import pandas as pd

In [189]:
# Get the FX rate EUR/USD dataset , range it and clean it
 
# Load file  - there is one single column
fx_data = pd.read_csv("data/EURUSD_D.csv", header=None)

# Rename the column
fx_data.columns=["Raw"]
# Remove the first row we do not need it
fx_data = fx_data.iloc[1:].reset_index(drop=True)

# Split using TAB separator (\t)
split_cols = fx_data["Raw"].str.split("\t", expand=True)

#  Build clean dataset without Time - only Date
fx_data_clean = pd.DataFrame({
    "Date": split_cols[0].str.slice(0, 10), 
    "Open": split_cols[1],
    "High": split_cols[2],
    "Low": split_cols[3],
    "Close": split_cols[4],
    "Volume": split_cols[5]
})

#  Convert Date in real date format
fx_data_clean["Date"] = pd.to_datetime(fx_data_clean["Date"], format="%Y-%m-%d")

# Convert numeric columns to float
cols = ["Open", "High", "Low", "Close", "Volume"]
fx_data_clean[cols] = fx_data_clean[cols].astype(float)


# Delete all the cols except Date and Close. Name Close to Excange Rate

fx_final = fx_data_clean[["Date", "Close"]]
fx_final.columns = ["Date", "Exchange rate"]

# Filter the needed period
fx_final = fx_final[
    (fx_final["Date"] >= "2016-03-24") &
    (fx_final["Date"] <= "2026-03-24")
]


# register the final dataset
fx_final.to_csv("data/EURUSD_worked.csv", index=False)

In [190]:
# Get the EU 2Y-bond Yield dataset, clean them and calculate their spread in a new dataset
# deleting the first rows because they are not necessary 
de2y = pd.read_csv("data/German2Y.csv", skiprows=9)

# delete rows with No value available
de2y = de2y[~de2y["Unnamed: 2"].astype(str).str.contains("No value available", na=False)]

# keep only two columns
de2y = de2y.iloc[:, :2]

#R ename the columns
de2y.columns=["Date", "Yield Rate"]
#  Convert Date in real date format
de2y["Date"] = pd.to_datetime(de2y["Date"], format="%Y-%m-%d")

# Convert numeric columns to float
cols = ["Yield Rate"]
# Clean Yield Rate
de2y["Yield Rate"] = (
    de2y["Yield Rate"]
    .astype(str)
    .str.replace("No value available", "")
    .str.replace(",", ".")
    .str.strip()
)

# Convert to numeric
de2y["Yield Rate"] = pd.to_numeric(de2y["Yield Rate"], errors="coerce")

# Drop invalid rows
de2y = de2y.dropna()




#keeps only rates for this period
de2y_final=de2y[
    (de2y["Date"] >= "2016-03-24") &
    (de2y["Date"] <= "2026-03-24")
]


de2y_final.to_csv("data/DE2Y_worked.csv", index=False)

In [191]:
# Get the US 2Y-bond Yield dataset, clean them and calculate their spread in a new dataset
us2y = pd.read_csv("data/US2Y.csv")

# Rename columns
us2y.columns = ["Date", "Yield Rate"]

#  Remove rows where Yield Rate is empty
us2y = us2y[us2y["Yield Rate"].notna()]

#  Convert Date
us2y["Date"] = pd.to_datetime(us2y["Date"], format="%Y-%m-%d")

#  Convert Yield Rate to numeric
us2y["Yield Rate"] = pd.to_numeric(us2y["Yield Rate"], errors="coerce")

# Drop any remaining bad rows
us2y = us2y.dropna()

#  Sort
us2y_final = us2y.sort_values("Date").reset_index(drop=True)

# Save clean file
us2y_final.to_csv("data/US2Y_worked.csv", index=False)


In [192]:
# Get the EU 10Y-bond Yield dataset, clean them and calculate their spread in a new dataset
# The same as the EU 2Y-bond Yield code , no worth creating a function as it repeats one time and not sure that the format is the same
# deleting the first rows because they are not necessary 
de10y = pd.read_csv("data/10Y_DE1.csv", skiprows=9)

# delete rows with No value available
de10y = de10y[~de10y["Unnamed: 2"].astype(str).str.contains("No value available", na=False)]

# keep only two columns
de10y = de10y.iloc[:, :2]

#Rename the columns
de10y.columns=["Date", "Yield Rate"]
#  Convert Date in real date format
de10y["Date"] = pd.to_datetime(de10y["Date"], format="%Y-%m-%d")

# Convert numeric columns to float
cols = ["Yield Rate"]
# Clean Yield Rate
de10y["Yield Rate"] = (de10y["Yield Rate"]
     .astype(str)
     .str.replace("No value available", "")
     .str.replace(",", ".")
     .str.strip()
 )

# Convert to numeric
de10y["Yield Rate"] = pd.to_numeric(de10y["Yield Rate"], errors="coerce")

# Drop invalid rows
de10y = de10y.dropna()




#keeps only rates for this period
de10y_final=de10y[
     (de10y["Date"] >= "2016-03-24") &
     (de10y["Date"] <= "2026-03-24")
 ]


de10y_final.to_csv("data/DE10Y_worked.csv", index=False)

In [193]:
# Get the US 10Y-bond Yield dataset, clean them and calculate their spread in a new dataset
# the same code as the 2Y - Bond Yield , not worth creating a separate function
us10y = pd.read_csv("data/US10Y.csv")

# Rename columns
us10y.columns = ["Date", "Yield Rate"]

#  Remove rows where Yield Rate is empty
us10y = us10y[us10y["Yield Rate"].notna()]

#  Convert Date
us10y["Date"] = pd.to_datetime(us10y["Date"], format="%Y-%m-%d")

#  Convert Yield Rate to numeric
us10y["Yield Rate"] = pd.to_numeric(us10y["Yield Rate"], errors="coerce")

# Drop any remaining bad rows
us10y = us10y.dropna()

#  Sort
us10y_final = us10y.sort_values("Date").reset_index(drop=True)

# Save clean file
us10y_final.to_csv("data/US10Y_worked.csv", index=False)

In [194]:
# create the spread 2y yield dataset
us2y_final=us2y_final.rename(columns= {"Yield Rate":"US_2Y"})
de2y_final=de2y_final.rename(columns={"Yield Rate":"DE_2Y"})
spread_yield=pd.merge(us2y_final, de2y_final, on="Date", how="inner")
spread_yield["Spread"]=spread_yield["US_2Y"]-spread_yield["DE_2Y"]
spread_yield = spread_yield.sort_values("Date").reset_index(drop=True)
spread_yield.to_csv("data/US_DE_2Y_spread.csv", index=False)

